In [2]:
!pip install transformers torch --quiet


In [5]:
!pip install gradio


   ---------------------------------------- 0.0/24.3 MB ? eta -:--:--
    --------------------------------------- 0.5/24.3 MB 5.6 MB/s eta 0:00:05
   - -------------------------------------- 0.8/24.3 MB 2.8 MB/s eta 0:00:09
   -- ------------------------------------- 1.3/24.3 MB 2.3 MB/s eta 0:00:10
   --- ------------------------------------ 1.8/24.3 MB 2.3 MB/s eta 0:00:10
   --- ------------------------------------ 2.1/24.3 MB 2.3 MB/s eta 0:00:10
   --- ------------------------------------ 2.4/24.3 MB 2.1 MB/s eta 0:00:11
   ---- ----------------------------------- 2.6/24.3 MB 2.0 MB/s eta 0:00:11
   ---- ----------------------------------- 2.6/24.3 MB 2.0 MB/s eta 0:00:11
   ---- ----------------------------------- 2.6/24.3 MB 2.0 MB/s eta 0:00:11
   ---- ----------------------------------- 2.6/24.3 MB 2.0 MB/s eta 0:00:11
   ---- ----------------------------------- 2.9/24.3 MB 1.3 MB/s eta 0:00:17
   ----- ---------------------------------- 3.1/24.3 MB 1.2 MB/s eta 0:00:18
   ---

In [6]:
from transformers import pipeline
import gradio as gr


In [7]:
qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")
print("✅ Model Loaded Successfully!")



config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

c:\Users\hp\.anaconda\conda\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--distilbert-base-cased-distilled-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

✅ Model Loaded Successfully!


In [8]:
def qa_chatbot_context_once():
    print("✅ QA Chatbot Started!")
    print("Commands: type 'change' to change context, type 'exit' to stop.\n")

    context = input("Enter Context Paragraph: ")

    while True:
        question = input("\nAsk Question: ")

        if question.lower() == "exit":
            print("✅ Chatbot Ended.")
            break

        if question.lower() == "change":
            context = input("\nEnter New Context Paragraph: ")
            continue

        result = qa_pipeline(question=question, context=context)

        answer = result["answer"]
        score = result["score"]

        # Confidence Threshold (No answer found handling)
        if score < 0.30:
            print("❌ Sorry, I could not find a confident answer in the given context.")
        else:
            print("✅ Answer:", answer)
            print("🔍 Confidence Score:", round(score, 4))


In [9]:
qa_chatbot_context_once()


✅ QA Chatbot Started!
Commands: type 'change' to change context, type 'exit' to stop.

✅ Answer: escape
🔍 Confidence Score: 0.8923
✅ Chatbot Ended.


In [13]:
def gradio_multi_qa(context, questions):
    if context.strip() == "":
        return "⚠️ Please enter context."

    if questions.strip() == "":
        return "⚠️ Please enter at least 1 question."

    # Split questions by new line
    question_list = [q.strip() for q in questions.split("\n") if q.strip() != ""]

    outputs = []
    for i, q in enumerate(question_list, start=1):
        result = qa_pipeline(question=q, context=context)
        answer = result["answer"]
        score = result["score"]

        if score < 0.30:
            outputs.append(f"Q{i}: {q}\n❌ Answer: Not found confidently\n🔍 Confidence: {round(score, 4)}\n")
        else:
            outputs.append(f"Q{i}: {q}\n✅ Answer: {answer}\n🔍 Confidence: {round(score, 4)}\n")

    return "\n".join(outputs)




In [14]:
demo = gr.Interface(
    fn=gradio_multi_qa,
    inputs=[
        gr.Textbox(lines=10, placeholder="Paste your context/paragraph here...", label="Context"),
        gr.Textbox(lines=8, placeholder="Type multiple questions (1 per line)\nExample:\nWhat is AI?\nWhere is AI used?\nWhat is Machine Learning?", label="Questions")
    ],
    outputs=gr.Textbox(lines=15, label="Answers"),
    title="📌 Multi-Question Answering Chatbot (Pre-trained Model)",
    description="Enter a context paragraph once, then ask multiple questions (one per line). The model will answer all questions together."
)


In [15]:
demo.launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
